In [ ]:
# 1. INSTALACIÓN DE LIBRERÍAS 
!pip install -q -U transformers datasets peft bitsandbytes trl accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from trl import SFTTrainer

# 2. CONFIGURACIÓN DEL MODELO Y DATASET
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
dataset_id = "Belenmrqz/genz-marketing-prompts" # Dataset de Hugging Face

# Carga del modelo en 4 bits 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token 

# 3. CONFIGURACIÓN DE LORA 
peft_config = LoraConfig(
    r=16,           # Rango (Capacidad del adaptador)
    lora_alpha=32,  # Factor de escala (Intensidad)
    target_modules=["q_proj", "v_proj"], # Capas del modelo a ajustar
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 4. CONFIGURACIÓN DEL ENTRENAMIENTO 
training_arguments = TrainingArguments(
    output_dir="./genz-model-output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    optim="paged_adamw_8bit",
    fp16=True, # Entrenamiento en 16 bits para velocidad
)

# 5. EJECUCIÓN DEL ENTRENAMIENTO 
dataset = load_dataset(dataset_id, split="train")

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="messages", # Campo JSONL
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_arguments,
)

print("¡Empezando el entrenamiento!")
trainer.train()

# 6. GUARDAR EL ADAPTADOR 
trainer.model.save_pretrained("./genz-adapter")
print("¡Modelo ajustado y guardado!")